In [1]:
import sys
sys.path.append('../')
from src.core.scraper.app import ScrapingUtils
from src.core.scraper.processor import ImagesProcessor
from src.core.scraper.utils import download_images, download_images_with_playwright
from src.utils.map_models_name import map_and_validate_model, get_brand_from_url
images_processor = ImagesProcessor()
scraper_utils = ScrapingUtils()

In [2]:
# model_data = ({"model": "V-Strom 1050 DE"}, None)
from typing import Optional, List, Any
from pydantic import BaseModel, Field

class ModelData(BaseModel):
    base_price: Optional[float] = Field(default=None)
    net_price: Optional[float] = Field(default=None)
    discount_amount: Optional[float] = Field(default=None)
    model: Optional[str] = Field(default=None)
    colors: Optional[List[str]] = Field(default=None)

In [3]:
# links = ["https://grupouma.com/colombia/motos/dominar/dominar-400-volcano/"]
# url = "https://grupouma.com/colombia/motos/pulsar/pulsar-ns200-fi-abs-dc/"

urls = ["https://www.yamaha-motor.com.mx/motos/street/fz-30-fi-2025"]
BRAND = "Yamaha"

extract_images = True
extract_technical_specs = True
extract_model_data = False

# Hardcode: crear el mismo tipo que devuelve el extractor
model_data = ModelData(model="FZ 3.0 FI")

### Verificar nombramiento

In [4]:
# for url in urls:
#     try:
#         print("Ejecutando extracción de model_data")
#         model_data, content = images_processor.get_model_data(url=url)
#         if model_data:
#             print("Hay model_data")
#         else:
#             print("No hay model_data")
#     except Exception as e:
#         print(f"Error extrayendo model_data: {e}")
#         model_data = None

#     try:
#         marca = get_brand_from_url(url)
#         model_data = map_and_validate_model(model_data, marca)
#     except ValueError as e:
#         print(f"\n[MAPEO] {e}")

## Ejecutar flujo

In [5]:
for url in urls:

    try:
        print(f"\nProcesando URL: {url}")

        if extract_model_data:
            try:
                print("Ejecutando extracción de model_data")
                model_data, content = images_processor.get_model_data(url=url)
                if model_data:
                    print("Hay model_data")
                else:
                    print("No hay model_data")
            except Exception as e:
                print(f"Error extrayendo model_data: {e}")
                model_data = None
                continue  # Si no hay model_data, no podemos continuar

            # Mapear nombre del modelo al nombre correcto del marketplace
            # Si no está mapeado o tiene valor vacío, se detiene el flujo con instrucciones
            try:
                marca = get_brand_from_url(url)
                model_data = map_and_validate_model(model_data, marca)
            except ValueError as e:
                print(f"\n[MAPEO] {e}")
                continue  # Detener procesamiento de esta URL hasta completar el mapeo

        if extract_images:
            try:
                print("Ejecutando extracción de imágenes")
                images = images_processor.get_images_from_website(url=url)
                print("Imageness")
                print(images)
                if images:
                    print(f"Total de imágenes: {len(images)}")
                else:
                    print("No hubo imágenes")
                # Descargar imágenes
                if model_data and hasattr(model_data, 'model'):
                    # Normalizar nombre para evitar problemas con espacios y caracteres especiales
                    safe_model_name = model_data.model.replace(" ", "_").replace("/", "_")
                    safe_model_name = model_data.model.replace(BRAND, "").strip()
                    safe_brand_name = BRAND.replace(" ", "_")
                    base_name = f"{safe_brand_name}_{safe_model_name}"
                    output_dir = f"../src/data/images/{safe_brand_name}_{safe_model_name}"

                    print(f"Intentando descargar {len(images)} imágenes...")
                    # Vento está protegido por Cloudflare Bot Challenge: requiere browser real
                    from src.core.scraper.processor import check_website
                    if check_website(url) == "vento":
                        results = download_images_with_playwright(images, base_name, output_dir, page_url=url, min_size_kb=10)
                    else:
                        results = download_images(images, base_name, output_dir, min_size_kb=10, page_url=url)

                    # Mostrar resultados
                    downloaded = sum(1 for r in results if r.get("ok", False) and not r.get("skipped", False))
                    skipped = sum(1 for r in results if r.get("ok", False) and r.get("skipped", False))
                    failed = sum(1 for r in results if not r.get("ok", False))
                    print(f"Descarga completada: {downloaded} descargadas, {skipped} omitidas (<100KB), {failed} fallidas")

                    if failed > 0:
                        print("Errores encontrados:")
                        for r in results:
                            if not r.get("ok", False):
                                print(f"  - {r.get('url', 'N/A')}: {r.get('error', 'Error desconocido')}")
                else:
                    print("No se puede descargar: model_data o model no disponible")
            except Exception as e:
                print(f"Error extrayendo imágenes: {e}")
                import traceback
                traceback.print_exc()
                images = None

        if extract_technical_specs:
            try:
                print("Ejecutando extracción de fichas técnicas")
                technical_specs = images_processor.get_technical_specs(url=url)
                if technical_specs:
                    print("Hay ficha técnica")
                    print(technical_specs)
                else:
                    print("No hay ficha técnica")
                # Guardar ficha técnica
                if model_data and hasattr(model_data, 'model'):
                    with open(f"../src/data/technical_specs/{BRAND} {model_data.model}.html", "w", encoding="utf-8") as f:
                        f.write(str(technical_specs))
                    print(f"Archivo HTML guardado como '{BRAND} {model_data.model}.html'")
            except Exception as e:
                print(f"Error extrayendo fichas técnicas: {e}")
                technical_specs = None
    except Exception as e:
        print(f"Error general en la URL {url}: {e}")
        continue


Procesando URL: https://www.yamaha-motor.com.mx/motos/street/fz-30-fi-2025
Ejecutando extracción de imágenes
url https://www.yamaha-motor.com.mx/motos/street/fz-30-fi-2025
website: yamaha
Imageness
['https://www.yamaha-motor.com.mx/assets/images/motorcycles/models/fz-30-fi-2025/3.jpg', 'https://www.yamaha-motor.com.mx/assets/images/motorcycles/models/fz-30-fi-2025/1.jpg', 'https://www.yamaha-motor.com.mx/assets/images/motorcycles/headers/fz-30-fi-2025.jpg', 'https://www.yamaha-motor.com.mx/assets/images/motorcycles/colors/fz-30-fi-2025/1.jpg', 'https://www.yamaha-motor.com.mx/assets/images/motorcycles/models/fz-30-fi-2025/2.jpg', 'https://www.yamaha-motor.com.mx/assets/images/motorcycles/colors/fz-30-fi-2025/2.jpg', 'https://www.yamaha-motor.com.mx/assets/images/motorcycles/models/fz-30-fi-2025/4.jpg']
Total de imágenes: 7
Intentando descargar 7 imágenes...
url https://www.yamaha-motor.com.mx/motos/street/fz-30-fi-2025
website: yamaha
Ejecutando descarga de imágenes... Total URLs: 7. 